# Unifying APIs with LiteLLM

Hello everyone! In the real world, the LLM landscape is vast and constantly changing. New, powerful models are released all the time from providers like Google, Anthropic, Meta, and more.

**This presents a major engineering challenge:** how do you use the best model for the job without rewriting your code every time? This is the problem that a brilliant open-source tool called **LiteLLM** solves.

LiteLLM is a **unified translation layer**. It allows you to call over 100 different LLM providers using the exact same format you've already learned. You write your code once, and LiteLLM handles the complex work of translating your request to the specific format each provider needs.

This notebook will demonstrate how to use LiteLLM to seamlessly switch between different AI providers.

## Installation and Setup

First, we need to install the `litellm` library. We also need to remember our security rule: each new provider will require its own API key, which must be set as an environment variable.

> Important: To run the final example in this notebook, you will need an API key from another provider, such as Anthropic (for Claude models). You would add this to your `.env` file just like you did for OpenAI.
> 
> 
> **Example `.env` file:**
> 
> ```
> OPENAI_API_KEY="sk-..."
> ANTHROPIC_API_KEY="sk-ant-..."
> 
> ```

In [17]:
import openai
import litellm
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

print("Libraries installed and environment variables loaded.")

Libraries installed and environment variables loaded.


In [31]:
OPENAI_MODEL="gpt-4o-mini"
MESSAGES = [
    {
    "role": "user",
    "content": "Tell me a fun fact about Golang."
    }
]

## The Baseline - A Standard OpenAI Call

Let's start with a standard API call using the `openai` library. This is our familiar baseline. We'll ask a simple question to see the standard response.

In [19]:
client = openai.OpenAI()
response_openai = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=MESSAGES
)

In [21]:
display(Markdown(response_openai.choices[0].message.content))

A fun fact about Golang, also known as Go, is that it was created by Google engineers Robert Griesemer, Rob Pike, and Ken Thompson and was officially announced in November 2009. One of the unique features of Go is its built-in support for concurrency, which is primarily facilitated through goroutines. A goroutine is a lightweight thread managed by the Go runtime, allowing developers to easily write programs that can perform multiple tasks simultaneously. This makes Go particularly well-suited for building scalable and efficient web servers and cloud services. In fact, you can spawn thousands of goroutines without the significant overhead traditionally associated with threads in other programming languages!

## LiteLLM Calling an OpenAI Model

Now, let's make the exact same call, but this time using `litellm`. Notice two things:

1. We call the `litellm.completion()` function.
2. The `model` string is now prefixed with the provider: `"openai/gpt-4o-mini"`.

The rest of the code structure is identical.

In [ ]:
# OpenAI models
valid_openai_models = litellm.get_valid_models(
    custom_llm_provider="openai",
    check_provider_endpoint=True
)
print(f"Valid OpenAI models: {valid_openai_models}")

Valid OpenAI models: ['gpt-4-turbo-2024-04-09', 'gpt-realtime-2.1-mini', 'gpt-realtime-2025-08-28', 'gpt-4', 'omni-moderation-2024-09-26', 'gpt-live-transcribe', 'gpt-5.6-luna', 'omni-moderation-latest', 'tts-1', 'gpt-5-mini', 'gpt-5-mini-2025-08-07', 'tts-1-hd', 'gpt-realtime-1.5', 'gpt-realtime', 'gpt-5.2-pro', 'gpt-5-chat-latest', 'gpt-3.5-turbo-0125', 'gpt-5.4-2026-03-05', 'gpt-5-nano', 'gpt-5.4-mini', 'o1-pro', 'gpt-5.1-codex', 'gpt-realtime-2', 'gpt-5-search-api', 'gpt-5.5-2026-04-23', 'gpt-5-2025-08-07', 'o4-mini-2025-04-16', 'gpt-4o-2024-11-20', 'gpt-5.1', 'whisper-1', 'gpt-5-pro', 'gpt-realtime-whisper', 'gpt-5.5-pro', 'gpt-audio-2025-08-28', 'gpt-4o-mini-transcribe-2025-03-20', 'gpt-4o-mini-transcribe-2025-12-15', 'tts-1-1106', 'gpt-5.5-pro-2026-04-23', 'gpt-audio-mini-2025-10-06', 'gpt-5.2-pro-2025-12-11', 'chat-latest', 'gpt-realtime-translate', 'gpt-5.4-pro-2026-03-05', 'gpt-5.4-mini-2026-03-17', 'babbage-002', 'gpt-4o-2024-05-13', 'text-embedding-3-large', 'gpt-4.1-nano',

In [ ]:
# Anthropic models
valid_anthropic_models = litellm.get_valid_models(
    custom_llm_provider="anthropic",
    check_provider_endpoint=True
)
print(f"Valid Anthropic models: {valid_anthropic_models}")

In [ ]:
PROVIDER = "openai"
# PROVIDER = "anthropic"
OPENAI_MODEL="gpt-4o-mini"
ANTHROPIC_MODEL = "claude-haiku-5-5"
MODEL = None
if PROVIDER == "openai":
    MODEL = OPENAI_MODEL
elif PROVIDER == "anthropic":
    MODEL = ANTHROPIC_MODEL

In [ ]:
response_litellm = litellm.completion(
    model=f"openai/{OPENAI_MODEL}",
    messages=MESSAGES
)

In [25]:
display(Markdown(response_litellm.choices[0].message.content))

A fun fact about Golang (or Go) is that it was developed at Google in just three years, starting in 2007, and was publicly announced in 2009. One of its main motivations was to address shortcomings in other programming languages used at Google, particularly in terms of compilation speed and ease of use for concurrent programming. The creators, Robert Griesemer, Rob Pike, and Ken Thompson, designed Go to be simple, efficient, and to make building scalable software easier—focusing on networked applications. As a result, it's widely used for cloud services, DevOps tools, and microservices!

## The Real Power - Switching to Another Provider

This is where LiteLLM's magic shines. We are now going to call a completely different model from a different company: Anthropic's Claude 3.5 Haiku.

The **only** thing we change in our code is the `model` string. We don't change the function call, the parameters, or the way we parse the response. As long as our `ANTHROPIC_API_KEY` is set in our `.env` file, LiteLLM handles everything.

In [30]:
valid_models = litellm.get_valid_models(
    custom_llm_provider="anthropic",
    check_provider_endpoint=True
)
print(valid_models)

['anthropic/claude-haiku-5-5', 'anthropic/claude-sonnet-5-5', 'anthropic/claude-opus-5-5', 'anthropic/claude-fable-5-1', 'anthropic/claude-opus-5', 'anthropic/claude-sonnet-5', 'anthropic/claude-fable-5', 'anthropic/claude-opus-4-8', 'anthropic/claude-opus-4-7', 'anthropic/claude-sonnet-4-6', 'anthropic/claude-opus-4-6', 'anthropic/claude-opus-4-5-20251101', 'anthropic/claude-haiku-4-5-20251001']


In [ ]:
response_litellm_claude = litellm.completion(
    model=f"anthropic/",
    messages=MESSAGES
)

In [29]:
display(Markdown(response_litellm_claude.choices[0].message.content))

Here's a fun one: Go's mascot is a gopher, designed by Renée French, and the gopher's name is often just "the Go gopher." Go was also designed at Google in 2007 by Robert Griesemer, Rob Pike, and Ken Thompson, and it became open source in 2009. Ken Thompson, one of the creators of Unix, also helped design the language, so it carries some of Unix's spirit.